In [ ]:
import numpy as np
import sys
sys.path.append("..")

from dataset_configs.smpl.utils import compute_B_from_beta, smpl_pose_to_D_orientation
from wimusim import WIMUSim, utils
import matplotlib.pyplot as plt

# Generating B and D Parameters from SMPL Pose Estimation

## Introduction
This notebook demonstrates how to generate **Body (B)** and **Dynamics (D)** parameters
from SMPL pose estimation output (e.g. **HMR2.0** or **4D-Humans**) and use them to
simulate virtual IMU data with WIMUSim.

- **Body (B)**: Limb lengths derived directly from SMPL shape parameters (β).
- **Dynamics (D)**: Per-joint relative orientations over time, converted from SMPL
  `global_orient` and `body_pose` rotation matrices.

### What We Will Do
1. **Load SMPL Output**: Load the pose estimation results saved as `.npz`.
2. **Compute B Parameters**: Use `compute_B_from_beta` to get bone vectors from β.
3. **Compute D Parameters**: Use `smpl_pose_to_D_orientation` to convert rotation matrices
   to WIMUSim joint quaternions.
4. **Define P and H Parameters**: Specify IMU placement and hardware defaults.
5. **Simulate Virtual IMU Data**: Run WIMUSim and visualize the output.

## 1. Load SMPL Pose Estimation Output

Run **HMR2.0** or **4D-Humans** on your video and save the output as a `.npz` file with keys:
- `betas`: shape parameters `(10,)` or `(1, 10)`
- `global_orient`: root rotation matrices `(T, 3, 3)`
- `body_pose`: per-joint rotation matrices `(T, 23, 3, 3)`

> **Note**: A sample output file can be generated using
> [4D-Humans](https://github.com/shubham-goel/4D-Humans) or
> [HMR2.0](https://github.com/shubham-goel/4D-Humans).

In [ ]:
# Load SMPL pose estimation output (from HMR2.0 / 4D-Humans)
# Save your output as .npz with keys:
#   betas:         shape parameters  (10,) or (1, 10)
#   global_orient: root rotation matrices  (T, 3, 3)
#   body_pose:     per-joint rotation matrices  (T, 23, 3, 3)
smpl_path = "data/smpl_output.npz"
data = np.load(smpl_path)

betas         = data["betas"]          # (10,) or (1, 10)
global_orient = data["global_orient"]  # (T, 3, 3)
body_pose     = data["body_pose"]      # (T, 23, 3, 3)

print(f"betas:         {betas.shape}")
print(f"global_orient: {global_orient.shape}")
print(f"body_pose:     {body_pose.shape}")

## 2. Compute Body (B) Parameters from β

The **Body (B)** parameters describe the limb lengths and bone vectors of the human body.
We derive them directly from the SMPL shape parameters (β) using `compute_B_from_beta`,
which runs a T-pose SMPL forward pass and returns the joint-to-joint bone vectors.

> **Note**: `smpl_model_path` should point to the directory containing `SMPL_NEUTRAL.pkl`
> (or gender-specific variants downloaded from the [SMPL website](https://smpl.is.tue.mpg.de/)).

In [ ]:
smpl_model_path = "path/to/smpl/models"  # directory containing SMPL_NEUTRAL.pkl

B_rp = compute_B_from_beta(betas, smpl_model_path=smpl_model_path, gender="neutral")
B_dict = {"rp": B_rp}

print("Bone vectors computed:", len(B_rp), "segments")
print("Sample entry:", list(B_rp.items())[0])

## 3. Compute Dynamics (D) Parameters

The **Dynamics (D)** parameters are the per-joint orientation time series.
`smpl_pose_to_D_orientation` converts SMPL rotation matrices to unit quaternions
in WXYZ format (PyTorch3D convention), one per joint per frame.

- `global_orient` → orientation of the root joint (`"BASE"`)
- `body_pose` → relative orientations for joints 1–23

In [ ]:
orientation = smpl_pose_to_D_orientation(global_orient, body_pose)
D_dict = {"orientation": orientation}

print("Joints:", list(orientation.keys()))
print("Frames:", list(orientation.values())[0].shape[0])

## 4. Define Placement (P) Parameters

The **Placement (P)** parameters specify where each virtual IMU is attached to the body.
Below we place a single IMU (`LLA`) on the left lower arm, offset from the `L_ELBOW` joint.

In [ ]:
P_dict = {
    "rp": {
        ("L_ELBOW", "LLA"): np.array([0.13, 0, 0.025]),
    },
    "ro": {
        ("L_ELBOW", "LLA"): np.deg2rad(np.array([0, 0, 180])),
    }
}

## 5. Define Hardware (H) Parameters

Hardware parameters model sensor noise and bias for each IMU.
We use `generate_default_H_configs` to create default values for the `LLA` IMU.

In [ ]:
H_dict = utils.generate_default_H_configs(["LLA"])
H_dict

## 6. Create a WIMUSim Environment and Simulate

With all four parameters defined (B, D, P, H), we initialise WIMUSim with
`dataset_name="SMPL"` and run the forward simulation to generate virtual IMU data.

In [ ]:
D_dict["sample_rate"] = 30  # HMR2.0 / 4D-Humans default output rate

wimusim_env = WIMUSim(B=B_dict, D=D_dict, P=P_dict, H=H_dict, dataset_name="SMPL", device="cpu")
virtual_IMU_dict = wimusim_env.simulate(mode="generate")

## 7. Visualize the Simulated IMU Data

In [ ]:
acc_LLA, gyro_LLA = virtual_IMU_dict["LLA"]

fig, axs = plt.subplots(3, 2, figsize=(10, 6))
start, end = 0, 300  # first 10 seconds at 30 Hz
axs[0, 0].set_title("Accelerometer")
axs[0, 1].set_title("Gyroscope")
for i in range(3):
    axs[i, 0].plot(acc_LLA[start:end, i].detach().cpu().numpy(), label=f"acc_LLA_{i}")
    axs[i, 1].plot(gyro_LLA[start:end, i].detach().cpu().numpy(), label=f"gyro_LLA_{i}")
    axs[i, 0].legend()
    axs[i, 1].legend()

plt.tight_layout()
plt.show()